# Robustness Evaluation: RoBERTa Predictions (Issue #18)

Loads the team's RoBERTa + LoRA detector (the "Logistic - Transformer" model from `02_logistic_3_transformer_training.ipynb`) and saves its predictions so the comparison notebook can use them.

1. Predict on the normal test set (the "before" numbers).
2. Predict on the paraphrased AI test set from issue #17 (the "after" numbers), once that file exists.

The model weights are not in the repo (gitignored). Put the `peft_model` and `tokenizer` folders in one folder, for example a `roberta_configs` folder in Google Drive, and set `MODEL_ROOT` below.

Run on a Colab GPU: Runtime → Change runtime type → T4 GPU.

## Setup
Clone the repo and install the packages.

In [ ]:
!git clone https://github.com/Break-Through-Tech/Chewy-1A-spot-the-bot-detecting-ai-generated-text.git
%cd Chewy-1A-spot-the-bot-detecting-ai-generated-text/notebooks
!pip -q install datasets huggingface_hub transformers peft accelerate

## Where the Model Files Are
Set `MODEL_ROOT` to the folder that contains `peft_model` and `tokenizer`. The next cell mounts Google Drive and checks that both folders are there.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

MODEL_ROOT = "/content/drive/MyDrive/roberta_configs"   # change if your folder is elsewhere
PEFT_MODEL_PATH = f"{MODEL_ROOT}/peft_model"
TOKENIZER_PATH = f"{MODEL_ROOT}/tokenizer"
PARAPHRASED_CSV = "../data/paraphrased_test.csv"        # from issue #17, change to its real path
PARAPHRASED_TEXT_COLUMN = "paraphrased_text"            # change to the real column name

import os
for path in (PEFT_MODEL_PATH, TOKENIZER_PATH):
    assert os.path.isdir(path), f"Missing folder: {path}. Ask Thu for her peft_model and tokenizer folders."
print("Model files found.")

## Load the Shared Test Split
Same test split every other model used (8,151 answers), so numbers are comparable.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
from spotbot.prepared import load_prepared_splits
from spotbot.evaluation import evaluate_predictions

_, _, test_df = load_prepared_splits()
print(test_df.shape)
print(test_df["label"].value_counts())

## Load the Model
Loads the saved LoRA weights on top of `roberta-base`, with the same 256-token limit used in training.

In [ ]:
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
from transformers import RobertaTokenizer
from peft import AutoPeftModelForSequenceClassification

MAX_LEN = 256
ID2LABEL = {0: "human", 1: "LLM-generated"}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = RobertaTokenizer.from_pretrained(TOKENIZER_PATH)
model = AutoPeftModelForSequenceClassification.from_pretrained(PEFT_MODEL_PATH, id2label=ID2LABEL)
model.to(device).eval()
print("Running on:", device)

## Prediction Function
Returns the model's probability that each text is AI-written. A text is called AI when that probability is at least 0.5.

In [ ]:
def predict_ai_prob(texts, batch_size=64):
    texts = [str(t) for t in texts]
    probs = []
    for start in tqdm(range(0, len(texts), batch_size)):
        batch = tokenizer(texts[start:start + batch_size], truncation=True,
                          max_length=MAX_LEN, padding=True, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = model(**batch).logits
        probs.append(torch.softmax(logits, dim=-1)[:, 1].cpu().numpy())
    return np.concatenate(probs)

## Original Test Set (the "before" numbers)
Should match Thu's result: macro-F1 about 0.99, accuracy about 0.991.

In [ ]:
orig_prob = predict_ai_prob(test_df["text"])
orig_pred = (orig_prob >= 0.5).astype(int)

orig_scores = evaluate_predictions(test_df["label"], orig_pred, test_df["source"])
print("Macro-F1:", round(orig_scores["macro_f1"], 4))
print("Accuracy:", round(orig_scores["accuracy"], 4))
print()
display(orig_scores["per_class"])
print("Confusion matrix (rows = true human/AI, columns = predicted human/AI):")
print(orig_scores["confusion_matrix"])
print()
print("Per-domain macro-F1:")
for domain, s in orig_scores["per_domain"].items():
    print(f"  {domain}: {s['macro_f1']:.4f}")

## Save the Original Predictions
One row per test answer. Send this file to Eliza for the comparison notebook.

In [ ]:
os.makedirs("../data/robustness", exist_ok=True)
orig_out = test_df[["question_group_id", "answer_index", "source", "label"]].copy()
orig_out["prob_ai"] = orig_prob
orig_out["pred"] = orig_pred
orig_out.to_csv("../data/robustness/original_test_predictions.csv", index=False)
print(orig_out.shape)
orig_out.head()

## Paraphrased AI Test Set (the "after" numbers)
Runs only if the #17 file exists at `PARAPHRASED_CSV`. That set contains only AI answers, so the number to watch is how many of them are still caught as AI (AI recall). The file needs a `label` column and the text column named above.

In [ ]:
if not os.path.exists(PARAPHRASED_CSV):
    print(f"Paraphrased file not found at {PARAPHRASED_CSV} yet (issue #17). Run this cell again once it exists.")
else:
    para_df = pd.read_csv(PARAPHRASED_CSV)
    assert PARAPHRASED_TEXT_COLUMN in para_df.columns, f"No column '{PARAPHRASED_TEXT_COLUMN}'. Columns: {list(para_df.columns)}"
    para_prob = predict_ai_prob(para_df[PARAPHRASED_TEXT_COLUMN])
    para_pred = (para_prob >= 0.5).astype(int)

    orig_ai_recall = orig_pred[test_df["label"].to_numpy() == 1].mean()
    print(f"AI answers caught: {orig_ai_recall:.4f} original  ->  {para_pred.mean():.4f} paraphrased")

    para_out = para_df.drop(columns=[PARAPHRASED_TEXT_COLUMN]).copy()
    para_out["prob_ai"] = para_prob
    para_out["pred"] = para_pred
    para_out.to_csv("../data/robustness/paraphrased_test_predictions.csv", index=False)
    print("Saved ../data/robustness/paraphrased_test_predictions.csv", para_out.shape)

## Download the Prediction Files
Downloads whichever prediction files exist so you can send them to Eliza or commit them.

In [ ]:
from google.colab import files
for name in ("original_test_predictions.csv", "paraphrased_test_predictions.csv"):
    path = f"../data/robustness/{name}"
    if os.path.exists(path):
        files.download(path)